<a href="https://colab.research.google.com/github/rekren/case_scrnaseq/blob/main/scRNAseq_case_study_R_and_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Technical case study: a single-cell RNA-seq perturbation experiment

---

## 1. What you have

One dataset from a **perturbation experiment**: immune cells from peripheral
blood, profiled by droplet-based single-cell RNA sequencing, under **two
conditions** (`ctrl` and `stim`), with cells from **eight donors** contributing
to both conditions.

Roughly 24,700 cells and 15,700 genes. `.X` holds **raw UMI counts**.

The object also carries a handful of columns and embeddings from **somebody
else's earlier analysis** of the same data. They are prefixed `provided_` when
the notebook loads. Treat them as a colleague's unreviewed work rather than as
ground truth: part of your job here is to decide how much of it you believe.

## 2. What we are actually looking for

Not package fluency. Neither of the roles we are hiring for assumes you have run
this workflow before, and every code cell already runs on sensible defaults, so
nobody gets stuck on syntax. The marks are in the written answers.

Three things in particular:

1. **Reasoning.** Say *why*, not *what*. "I set the resolution to 0.5" is worth
   nothing; "I set it to 0.5 because 1.0 split cluster 3 without producing any
   distinguishing marker gene" is worth a great deal. For every conclusion, name
   the observation that would have pointed the other way.
2. **Curiosity.** Several things in this dataset are odd. Some are deliberate,
   some are properties of the data itself, and at least one will not match what
   any tutorial told you to expect. We are more interested in whether you
   *notice* and *chase* them than in whether you resolve them.
3. **Carrying your own modality across.** You each come from a different area of
   omics. Single-cell transcriptomics borrows heavily from both, and breaks from
   both in specific places. Several questions ask you explicitly to map a concept
   from the modality you know onto this one, and to say where the analogy stops
   working. Answering these well does not require any single-cell background. It
   requires you to think clearly about what you already know.

**Speculate, and label it as speculation.** Most of these questions cannot be
settled from the data alone. That is deliberate. "My best interpretation is X,
because Y, and I would test it by Z" is exactly the shape of answer we want. A
labelled, well-argued guess scores higher than silence and much higher than
false confidence. "I do not know, and here is how I would find out" is a
complete answer. Inventing a marker gene or a citation is not.

Wrong answers defended with clear evidence score above right answers with no
justification.

## 3. About the source of the data

This is a public dataset. You may well recognise it, and if you do, that is
fine and costs you nothing.

**Please do not go looking for it, and do not read the original paper.** We are
not testing whether you can retrieve a published answer, and an analysis
reverse-engineered from someone else's figures tells us nothing about how you
think. If you recognise it, say so in Q22 and carry on reasoning from the
matrix. That is all we want to see.

## 4. Choose one track

This notebook contains the same analysis twice.

| | Section 1 | Section 2 |
|---|---|---|
| Language | R, `Seurat` | Python, `scanpy` |
| Colab runtime | **R** | **Python 3** |
| File it reads | `case_data.RDS` | `case_data.h5ad` |

**Work through one section and ignore the other.** The questions are identical
in both, under the same numbers Q1 to Q23, and they are marked identically. Set
`Runtime > Change runtime type` before running anything. If you pick R, the
Python cells will not run, and that is expected.

The choice is not scored and there is no hidden preference. For context only:
the group is R-fluent and most routine analysis here is written in R, so R is a
natural default and you would be well supported in it. Python is also valued,
and not only for single-cell work. We would rather read clear reasoning in your
stronger language than hesitant code in your weaker one.

## 5. Ground rules

1. **Time.** Approximately 3 hours of focused work, plus an hour for the slides in
   Part 9. If you run out of time, stop and write down what you would have done
   next. That answer is itself marked.
2. **Every question needs an answer.** Two to six sentences. An empty box scores
   zero, and so does "used the default". If you tried something that failed, say
   so; that is a valid answer.
3. **Use any tool you like**, including documentation, forums and language
   models, with the single exception in section 3 above. If a tool wrote part of
   your answer, say which part and whether you checked it (Q22).

## 6. What to send back

- This notebook, executed, **with outputs visible** (`File > Download > .ipynb`).
- The annotation table from Part 6.
- The one-page summary from Part 8.
- The short slide deck from Part 9.

## 7. Question index

| Part | Questions | Topic |
|---|---|---|
| 1 | Q1 to Q3 | First contact: what is raw, what is somebody's conclusion |
| 2 | Q4 to Q6 | Quality control, including one metric you cannot compute |
| 3 | Q7, Q8 | Normalisation and feature selection |
| 4 | Q9 to Q11 | Dimensionality reduction and clustering |
| 5 | Q12 to Q14 | **The perturbation. The core of the exercise.** |
| 6 | Q15 to Q17 | Cell identity |
| 7 | Q18, Q19 | Differential expression done properly |
| 8 | Q20 to Q22 | Synthesis, critique, honest reporting |
| 9 | Q23 | Presentation |

Short on time? Prioritise Parts 1, 2, 5 and 7, then write the synthesis. Parts 4
and 9 are the ones to drop first.

---


---
---

# SECTION 1. R track (`Seurat`)

**Work here if you chose R.** Set `Runtime > Change runtime type` to
**R** first. Questions Q1 to Q23 are identical to the other section.

---

## Part 0. Environment  *(R track)*


In [ ]:
# --- Colab R runtime setup -------------------------------------------------
# Posit Public Package Manager serves precompiled Linux binaries, which turns a
# 30-minute source build of Seurat into a few minutes. The HTTPUserAgent option
# is what tells the mirror to serve binaries; do not drop it. The repository URL
# is keyed to the Ubuntu release, so detect it rather than hard-coding it.

codename <- tryCatch({
  os  <- readLines("/etc/os-release", warn = FALSE)
  hit <- grep("^UBUNTU_CODENAME=", os, value = TRUE)
  if (length(hit)) sub("^UBUNTU_CODENAME=", "", hit[1]) else "jammy"
}, error = function(e) "jammy")
message("Ubuntu codename: ", codename)

options(
  repos = c(P3M = sprintf(
    "https://packagemanager.posit.co/cran/__linux__/%s/latest", codename)),
  HTTPUserAgent = sprintf("R/%s R (%s)", getRversion(),
    paste(getRversion(), R.version$platform, R.version$arch, R.version$os)),
  Ncpus = max(1L, parallel::detectCores()), timeout = 1800
)

invisible(system("apt-get -qq update > /dev/null 2>&1"))
invisible(system(paste("apt-get -qq install -y libglpk40 libxml2-dev libhdf5-dev",
                       "libfontconfig1-dev libharfbuzz-dev libfribidi-dev",
                       "> /dev/null 2>&1")))

# CRAN only, on purpose. Nothing here needs Bioconductor: its source mirror
# is slow and unreliable from Colab, and a single 504 there would strand you in
# a 12-package source build.
pkgs <- c("Seurat", "SeuratObject", "Matrix", "ggplot2", "patchwork", "dplyr")
need <- setdiff(pkgs, rownames(installed.packages()))
if (length(need)) install.packages(need)
still <- setdiff(pkgs, rownames(installed.packages()))
if (length(still)) install.packages(still, repos = "https://cloud.r-project.org")

missing <- setdiff(pkgs, rownames(installed.packages()))
if (length(missing)) {
  stop("Could not install: ", paste(missing, collapse = ", "),
       "
Rerun this cell; the binary mirror occasionally times out.")
}

suppressPackageStartupMessages({
  library(Seurat); library(Matrix); library(ggplot2)
  library(patchwork); library(dplyr)
})
set.seed(0)
options(repr.plot.width = 12, repr.plot.height = 5)
message("Setup finished. Seurat ", packageVersion("Seurat"))


In [ ]:
# FALLBACK ONLY. Skip this cell if the R runtime worked.
# If Colab does not offer an R runtime, start a Python runtime, run this cell,
# then prefix every subsequent R cell with %%R (and size figures with
# e.g. %%R -w 900 -h 420 -r 100).
%load_ext rpy2.ipython


---

## Part 1. First contact

Before running a single transformation, work out what you are holding.

A real object arrives with history. Some of what is in `obs` is measurement,
some is a previous analyst's conclusion, and telling those apart is the first
skill this exercise tests.  *(R track)*


In [ ]:
# ---------------------------------------------------------------------------
# The dataset downloads itself. Nothing to configure.
#
# If the download fails (institutional network, proxy, host down), fetch the
# file in a browser and upload it via the left sidebar: Files > Upload. The cell
# then picks it up automatically on rerun.
# ---------------------------------------------------------------------------
DATA_URL <- "https://web-genobioinfo.toulouse.inrae.fr/~rekren/case_data.RDS"
PATH     <- "case_data.RDS"

if (!file.exists(PATH)) {
  options(timeout = 3600)
  ok <- tryCatch({ download.file(DATA_URL, PATH, mode = "wb"); TRUE },
                 error = function(e) { message("Download failed: ", conditionMessage(e)); FALSE })
  if (!ok || !file.exists(PATH)) {
    stop("Could not download the data. Fetch it in a browser from\n  ", DATA_URL,
         "\nthen upload it via the left sidebar (Files > Upload) and rerun this cell.")
  }
}
if (file.size(PATH) < 1e6) {
  unlink(PATH)
  stop("The downloaded file was too small to be the data, so the link returned ",
       "an error page. Deleted it. Download in a browser and upload instead.")
}
cat("data file:", round(file.size(PATH) / 1e6, 1), "MB\n")

# A plain list: $counts (genes x cells, dgCMatrix), $meta (data.frame),
# $reductions (the earlier analysis's embeddings). Needs only Matrix.
obj <- readRDS(PATH)
str(obj, max.level = 1)

meta <- obj$meta

# Rename to a stable vocabulary, and mark the columns that are somebody else's
# conclusions rather than measurements.
ren <- c(label = "condition", stim = "condition",
         replicate = "donor", ind = "donor",
         cell_type = "provided_cell_type", cell = "provided_cell_type",
         seurat_clusters = "provided_cluster")
for (old in names(ren)) if (old %in% colnames(meta)) {
  colnames(meta)[colnames(meta) == old] <- ren[[old]]
}

so <- CreateSeuratObject(counts = obj$counts, meta.data = meta, project = "case",
                         min.cells = 0, min.features = 0)

# Carry the earlier analysis's embeddings across, clearly labelled.
for (rd in names(obj$reductions)) {
  emb <- obj$reductions[[rd]]
  colnames(emb) <- paste0(rd, "_", seq_len(ncol(emb)))
  so[[paste0("provided_", rd)]] <- CreateDimReducObject(
    embeddings = emb[colnames(so), , drop = FALSE],
    key = paste0("prov", rd, "_"), assay = "RNA")
}

so


In [ ]:
# What is in here? Extend these checks as much as you like.
cat("cells x genes:", ncol(so), "x", nrow(so), "\n\n")

cat("--- metadata columns ---\n")
for (cn in colnames(so@meta.data)) {
  v <- so@meta.data[[cn]]
  if (is.numeric(v) && length(unique(v)) > 20) {
    cat(sprintf("  %-24s numeric   range %.1f to %.1f\n", cn, min(v), max(v)))
  } else {
    lv <- unique(as.character(v))
    cat(sprintf("  %-24s %2d levels: %s\n", cn, length(lv),
                paste(head(lv, 8), collapse = ", ")))
  }
}

cat("\n--- are the values raw counts? ---\n")
m <- LayerData(so, assay = "RNA", layer = "counts")[, 1:200]
cat("integer-valued:", all(m@x == round(m@x)), "\n")
cat("range of non-zero values:", min(m@x), "to", max(m@x), "\n")
cat("fraction of zeros:", round(1 - length(m@x) / prod(dim(m)), 4), "\n")

cat("\n--- the design ---\n")
print(table(so$condition))
print(table(so$donor, so$condition))

cat("\n--- reductions already present ---\n")
print(Reductions(so))


### Q1
Describe what you have been given. Cover the dimensions, what a row and a column
represent, what the values in the matrix are, and how you *verified* they are
raw counts rather than something already transformed.

> **Your answer:**
>
> _(replace this line)_


### Q2
Go through the metadata columns one by one and sort them into three groups:

- **measured**: a property of the experiment or the sequencing
- **derived**: something computed from the counts, which you could recompute
- **concluded**: somebody's interpretation, which you could disagree with

Say which group each column falls into and how you decided. Then: which of these
columns would you be comfortable using in your own analysis, and which would you
set aside until you have checked them yourself?

> **Your answer:**
>
> _(replace this line)_


### Q3
The matrix is about 96 % zeros.

**(a)** Give two distinct reasons for that, one technical and one biological.

**(b)** **Translate from your own modality.** If your background is bulk
RNA-seq: a bulk count matrix has almost no zeros, and this one is almost all
zeros, yet both are counting the same molecules. Explain what changed. If your
background is proteomics: you already deal with missing values constantly.
Compare a zero here with a missing value in a proteomics run. Are they the same
kind of nothing? What follows for imputation?

> **Your answer:**
>
> _(replace this line)_


---

## Part 2. Quality control

Droplet data contains barcodes that are not single healthy cells: near-empty
droplets, dying cells, debris, and droplets that captured more than one cell.
This part is about deciding which barcodes to keep.

One warning before you start: at least one thing in this part will not match
what the tutorials tell you to do. Finding out what, and reasoning about it, is
the question.

*(R track)*


In [ ]:
# Gene classes that carry diagnostic information about cell quality.
g <- rownames(so)

mito <- grep("^MT-", g, value = TRUE)          # the standard first filter
ribo <- grep("^RP[SL]", g, value = TRUE)
hb   <- grep("^HB[ABDGQ][0-9]?$", g, value = TRUE)

cat("mitochondrial genes found:", length(mito), "\n")
cat("ribosomal genes found:    ", length(ribo), "\n")
cat("haemoglobin genes found:  ", length(hb), "\n\n")

if (length(mito)) {
  so[["percent_mt"]] <- PercentageFeatureSet(so, features = mito)
} else {
  cat(">>> Read that first number again, then answer Q5.\n",
      "    Do not skip past this. It changes what you can do next.\n\n")
}
so[["percent_ribo"]] <- PercentageFeatureSet(so, features = ribo)
if (length(hb)) so[["percent_hb"]] <- PercentageFeatureSet(so, features = hb)

qc_cols <- intersect(c("nCount_RNA", "nFeature_RNA", "percent_mt",
                       "percent_ribo", "percent_hb"), colnames(so@meta.data))
summary(so@meta.data[, qc_cols])


### Q4
For each metric below, say in one sentence what a cell with an extreme value of
it most likely is, and be specific about the direction.

| metric | what an extreme value most likely means |
|---|---|
| total counts per cell | *your answer* |
| genes detected per cell | *your answer* |
| ribosomal read fraction | *your answer* |
| haemoglobin read fraction | *your answer* |

Then: total counts and genes detected are strongly correlated. Why look at both?

> **Your answer:**
>
> _(replace this line)_


### Q5
Every standard quality-control workflow for this data type leans on one
particular metric, and you have just tried to compute it.

**(a)** Which metric, what does a high value of it indicate, and why is it the
usual first filter?

**(b)** What happened when you tried to compute it here? Report what you found,
not what you expected.

**(c)** Speculate about why. Give at least two possible explanations and say
which you find more plausible and why. You will not be able to confirm it, and
you are not expected to.

**(d)** Given that it is unavailable, what do you use instead? Name the
substitute metrics you will rely on and say what each one does and does not
capture that the missing one would have.

> **Your answer:**
>
> _(replace this line)_


In [ ]:
options(repr.plot.width = 14, repr.plot.height = 4.5)
VlnPlot(so, features = qc_cols, group.by = "condition",
        pt.size = 0, ncol = length(qc_cols)) & theme(legend.position = "none")

FeatureScatter(so, "nCount_RNA", "nFeature_RNA", group.by = "condition") |
  FeatureScatter(so, "nCount_RNA", "percent_ribo", group.by = "condition")


In [ ]:
# TODO: choose and justify your thresholds (Q6).
# A median-absolute-deviation helper, in case you prefer data-driven cutoffs.
mad_outlier <- function(x, n_mads = 5, log = FALSE) {
  if (log) x <- log1p(x)
  m <- median(x); d <- median(abs(x - m))
  (x < m - n_mads * d) | (x > m + n_mads * d)
}

MIN_FEATURES <- NULL   # e.g. 200
MAX_FEATURES <- NULL   # e.g. 2500
MAX_RIBO     <- NULL   # e.g. 50

# keep <- ...
# cat("keeping", sum(keep), "of", ncol(so), "cells\n")
# so_raw <- so
# so <- so[, keep]
# so


### Q6
State every filtering threshold you applied and the evidence in the plots above
that led to each number. "The tutorial used it" is not evidence.

Then argue against yourself:

**(a)** How many cells did you remove, in absolute numbers and as a percentage?
Is that plausible?

**(b)** Which real immune cell type is most likely to be discarded by your
threshold? Think about cells with unusually little RNA.

**(c)** You probably also filtered genes. That reduces noise, but it is not
free. What do you lose?

> **Your answer:**
>
> _(replace this line)_


---

## Part 3. Normalisation and feature selection

Two cells can produce different total counts purely because of capture
efficiency rather than biology.

*(R track)*


In [ ]:
# TODO: normalise, select variable features, scale (Q7, Q8).
N_HVG <- NULL   # e.g. 2000

# so <- NormalizeData(so, normalization.method = "LogNormalize", scale.factor = 1e4)
# so <- FindVariableFeatures(so, selection.method = "vst", nfeatures = N_HVG)
# head(VariableFeatures(so), 30)
# VariableFeaturePlot(so) |> LabelPoints(points = head(VariableFeatures(so), 15), repel = TRUE)


### Q7
**(a)** Explain what the normalisation you applied actually does, step by step.
If it has two steps, say what problem *each* one solves; they are different
problems.

**(b)** **Translate from your own modality.** From bulk RNA-seq: compare this
with CPM, TMM, or DESeq2 size factors. Name one assumption that carries over and
one that does not. From proteomics: compare it with how you normalise intensity
data across runs, and say which of the two situations you consider better posed
and why.

**(c)** A per-cell scaling factor assumes something about the cells being
compared. What does it assume, and can you think of a situation in immunology
where that assumption is clearly wrong?

> **Your answer:**
>
> _(replace this line)_


### Q8
**(a)** Why select a subset of genes at all? What specifically goes wrong if you
keep all of them?

**(b)** How many did you keep, and how did you land on that number? What would
change with 500 instead, or 5000?

**(c)** Look at the top-ranked variable genes. Do any of them look like they
track a technical or unwanted process rather than cell identity? Name them and
say what you suspect.

> **Your answer:**
>
> _(replace this line)_


---

## Part 4. Dimensionality reduction and clustering

You will produce your own embedding and your own clusters. The object already
contains an embedding and a clustering from the earlier analysis. Do not look at
them until you have made your own; then compare.

*(R track)*


In [ ]:
# TODO: scale and run PCA (Q9).
# so <- ScaleData(so, features = VariableFeatures(so), verbose = FALSE)
# so <- RunPCA(so, features = VariableFeatures(so), npcs = 50, verbose = FALSE)
# ElbowPlot(so, ndims = 50)
# print(so[["pca"]], dims = 1:3, nfeatures = 12)
# DimPlot(so, reduction = "pca", group.by = "condition")


### Q9
**(a)** Why run PCA before clustering rather than clustering the genes directly?
Give two distinct reasons.

**(b)** How many components did you carry forward, and on what basis? What is
the practical consequence of taking too few? Of taking too many?

**(c)** Look at the top gene loadings on the first two or three components. For
each, say what biological or technical process you think it is capturing, naming
the genes you are relying on.

> **Your answer:**
>
> _(replace this line)_


In [ ]:
# TODO: neighbours, UMAP, clustering at several resolutions (Q10).
N_PCS <- NULL   # from Q9

# so <- FindNeighbors(so, dims = 1:N_PCS)
# for (r in c(0.2, 0.5, 1.0)) so <- FindClusters(so, resolution = r, verbose = FALSE)
# so <- RunUMAP(so, dims = 1:N_PCS, verbose = FALSE)
# DimPlot(so, group.by = c("RNA_snn_res.0.2", "RNA_snn_res.0.5", "RNA_snn_res.1"),
#         label = TRUE, ncol = 3)
# DimPlot(so, group.by = "condition")


### Q10
**(a)** A UMAP is an embedding and Leiden is a clustering algorithm, and they run
on the same graph but are not the same thing. Explain the relationship. In
particular: if two clusters sit next to each other on the UMAP, does that mean
they are biologically similar?

**(b)** Which resolution did you settle on, and what evidence supports it?
Raising the resolution always yields more clusters, so what stops this being
arbitrary? Describe one concrete check you ran, or would run, to decide whether
a split is real.

> **Your answer:**
>
> _(replace this line)_


In [ ]:
# Compare your clustering with the one already in the object (Q11).
MY_CLUSTERS <- NULL   # e.g. "RNA_snn_res.0.5"

# cat("--- your clusters vs condition ---\n")
# print(round(prop.table(table(so@meta.data[[MY_CLUSTERS]], so$condition), 1), 2))
#
# cat("\n--- provided clusters vs condition ---\n")
# print(round(prop.table(table(so$provided_cluster, so$condition), 1), 2))
#
# cat("\n--- your clusters vs provided clusters ---\n")
# print(table(so@meta.data[[MY_CLUSTERS]], so$provided_cluster))
#
# if (requireNamespace("mclust", quietly = TRUE)) {
#   cat("\nadjusted Rand index:",
#       mclust::adjustedRandIndex(so@meta.data[[MY_CLUSTERS]], so$provided_cluster), "\n")
# }
#
# DimPlot(so, group.by = "provided_cell_type", label = TRUE) |
#   DimPlot(so, reduction = "provided_umap", group.by = "condition")


### Q11
Now compare your clustering with the one already in the object.

**(a)** Do they agree? Quantify it, do not just eyeball the plots.

**(b)** Cross-tabulate your clusters against `condition`, and then the provided
clusters against `condition`. The two tables look very different. Describe the
difference precisely.

**(c)** **Speculate.** What must the earlier analyst have done that you did not?
You have enough information to work this out from the tables alone. State your
reasoning.

**(d)** Which of the two clusterings is *correct*? Argue it. This is a real
question, not a rhetorical one, and there is a defensible case on each side.

> **Your answer:**
>
> _(replace this line)_


---

## Part 5. The perturbation

This is the core of the exercise. Take your time here, at the cost of Part 4 or
Part 9 if necessary.

The dataset has two conditions and eight donors. Every donor contributes cells
to **both** conditions.

*(R track)*


### Q12
Before running anything else in this part, write down the question you think
this experiment was designed to answer, based only on what you have seen so far.
One or two sentences. Do not edit it afterwards.

> **Your answer:**
>
> _(replace this line)_


In [ ]:
# Tools for Part 5. Use, modify or ignore.

# 1. Which genes separate the conditions within a single cell population?
# Idx <- WhichCells(so, expression = provided_cell_type == "<pick one>")
# sub <- so[, Idx]
# Idents(sub) <- sub$condition
# de <- FindMarkers(sub, ident.1 = "stim", ident.2 = "ctrl", logfc.threshold = 0.25)
# head(de[order(-de$avg_log2FC), ], 25)

# 2. Score every cell for the genes you found, then look at where the score sits.
# my_genes <- list(programme = c("<gene>", "<gene>", "..."))
# so <- AddModuleScore(so, features = my_genes, name = "programme")
# VlnPlot(so, "programme1", group.by = "provided_cell_type", split.by = "condition",
#         pt.size = 0)

# 3. The paired design: does the effect hold within every donor?
# so@meta.data |>
#   group_by(donor, condition) |>
#   summarise(score = mean(programme1), .groups = "drop") |>
#   tidyr::pivot_wider(names_from = condition, values_from = score)


### Q13
You now have to decide whether the separation between `ctrl` and `stim` is a
**technical batch effect that should be removed** or a **biological difference
that must be preserved**. Removing it in the first case is correct; removing it
in the second destroys the finding.

**(a)** Design the test *before* you run it. What would you measure, and what
result would point which way? Write this down first.

**(b)** Run it. Report what you found.

**(c)** The eight donors are the strongest tool you have here, because the
design is paired. Explain why, and use them.

**(d)** Commit to an answer and defend it with specific genes. Then say what
observation would have changed your mind.

> **Your answer:**
>
> _(replace this line)_


### Q14
**(a)** Name the genes that dominate the difference between conditions. Do they
form a coherent programme? If you recognise it, name it. If you do not, describe
what they have in common as far as you can tell from the data, and say how you
would identify it.

**(b)** Compute the effect separately within each cell type. Is the response the
same size everywhere? Report the ranking.

**(c)** **This is the most interesting result in the dataset.** One population
responds far less than the others. Identify it, and give your best explanation.
Speculate freely here, and say how confident you are.

> **Your answer:**
>
> _(replace this line)_


---

## Part 6. Cell identity

The object contains a `provided_cell_type` column. **Do your own annotation
first**, from marker genes, without looking at it. Then compare.

We will be able to tell from the notebook's execution order whether you did this
in the stated order, and answering honestly in Q17 matters more to us than
matching the provided labels.

*(R track)*


### Q15
Describe your annotation procedure as you would in a methods section: how did you
get from a ranked gene list to a label?

Then fill in the table, one row per cluster, at your chosen resolution.

| cluster | n cells | your label | supporting genes (3 to 5) | confidence | what would raise it |
|---|---|---|---|---|---|
| | | | | | |

*(add rows as needed)*

> **Your answer:**
>
> _(replace this line)_


### Q16
**(a)** Which cluster are you least sure about, and what are the two competing
interpretations? Do not resolve the ambiguity artificially; describe it.

**(b)** Is there a cluster you suspect is not a real cell type at all? What
positive evidence would confirm that, and what would refute it?

**(c)** **Translate from your own modality.** From proteomics, and particularly
spatial proteomics: you are used to identifying populations from a small panel
of markers chosen in advance. Here you have every gene but a noisy, sparse
measurement of each. Compare the two situations. Which errors does each protect
you from, and which does each expose you to? From bulk RNA-seq: you are used to
deconvolution or signature scoring against a reference. How does assigning
identity cell by cell change what you can claim?

> **Your answer:**
>
> _(replace this line)_


### Q17
Now compare with `provided_cell_type`.

**(a)** Where do you agree and where do you differ? Quantify it.

**(b)** For each disagreement, say who you think is right and why.

**(c)** Did looking at the provided labels change your mind about any cluster?
Say so plainly if it did. Changing your mind on evidence is a good sign;
quietly rewriting an earlier answer is not.

> **Your answer:**
>
> _(replace this line)_


---

## Part 7. Differential expression done properly

Suppose you now want to report which genes respond to the perturbation in a
given cell type, in a form a colleague could act on.

*(R track)*


In [ ]:
# Part 7. Cell-level test first, then aggregate.

# --- cell level ---
# sub <- so[, so$provided_cell_type == "<pick one>"]
# Idents(sub) <- sub$condition
# de_cell <- FindMarkers(sub, ident.1 = "stim", ident.2 = "ctrl")
# sum(de_cell$p_val_adj < 0.05)

# --- aggregated to one profile per donor per condition ---
# pb <- AggregateExpression(so, assays = "RNA", slot = "counts",
#                           group.by = c("provided_cell_type", "donor", "condition"))
# dim(pb$RNA)
# Take one cell type, build a matrix of donors x conditions, and test with
# limma/edgeR/DESeq2 if available, or a paired test per gene if not.


### Q18
The obvious approach is to take all cells of one type, split them by condition,
and run a test across individual cells.

**(a)** Do that, and report the number of genes passing a conventional
significance threshold.

**(b)** Now criticise the result you just produced. What is wrong with treating
each cell as an independent observation here? Name the specific statistical
problem.

**(c)** How does the number in (a) compare with the number of *donors*? What
does that comparison tell you?

> **Your answer:**
>
> _(replace this line)_


### Q19
**(a)** Aggregate the counts to one profile per donor per condition per cell
type, and analyse that instead. Report what changes.

**(b)** **Translate from your own modality.** If your background is bulk
RNA-seq, you have just rebuilt something very familiar. Say what it is, and why
the single-cell field spent years rediscovering it. If your background is
proteomics, describe the equivalent aggregation you would perform, and what your
replicate unit is.

**(c)** What did you lose by aggregating? Name something the cell-level analysis
could have told you that the aggregated one cannot.

> **Your answer:**
>
> _(replace this line)_


---

## Part 8. Synthesis

*(R track)*


### Q20
In at most one page, state what is in this dataset: the populations, the
structure, and what you believe the two conditions represent. Write it for a
biologist who will act on it, not for a bioinformatician.

> **Your answer:**
>
> _(replace this line)_


### Q21
**(a)** Which single decision in this notebook had the largest influence on your
conclusions? How would they change if you had chosen differently?

**(b)** Which parts of your answer would you defend in a lab meeting, and which
are provisional?

**(c)** With a month instead of an afternoon, what would you do first, second and
third, and what question would each answer?

**(d)** List every assumption you made that you could not verify.

> **Your answer:**
>
> _(replace this line)_


### Q22
**(a)** Which external resources did you use, including documentation, forums and
language models, and for what specifically? Was there any output you accepted
without being able to verify it?

**(b)** Which cell did you run without fully understanding? There is almost
always one, and naming it scores better than pretending otherwise.

**(c)** Did you recognise the dataset? If so, at what point and from what? This
carries no penalty.

**(d)** What surprised you most, coming from your own area of omics? What was
harder than you expected, and what turned out to be easier?

> **Your answer:**
>
> _(replace this line)_


---

## Part 9. Presentation

Prepare **five to eight slides** for a 15 minute presentation to the group,
followed by discussion. Assume an audience of immunologists and one
bioinformatician.

Suggested shape, which you may ignore:

1. What the data was and how you approached it
2. Quality control, and the metric you could not compute
3. The populations you found
4. The perturbation: what it does, and to which cells most
5. One thing you are unsure about, and how you would resolve it

We would rather see one honest slide about uncertainty than five confident ones.

*(R track)*


### Q23 (deliverable, not a written answer)
Attach the slides with your notebook. No template, any format.

> **Anything you want us to know about the slides:**
>
> _(replace this line)_


In [ ]:
# Environment record. Run this last and leave the output in place.
sessionInfo()


---
---

# SECTION 2. Python track (`scanpy`)

**Work here if you chose Python.** Set `Runtime > Change runtime type` to
**Python 3** first. Questions Q1 to Q23 are identical to the other section.

---

## Part 0. Environment  *(Python track)*


In [ ]:
# Colab setup. Two to four minutes on a fresh runtime.
%pip install -q "scanpy>=1.10" "anndata>=0.10" leidenalg igraph scikit-misc

import warnings
warnings.simplefilter("ignore", FutureWarning)

import scanpy as sc, anndata as ad
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import scipy.sparse as sp

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=80, facecolor="white")
SEED = 0
np.random.seed(SEED)

from importlib.metadata import version
print("scanpy", version("scanpy"), "| anndata", version("anndata"))


---

## Part 1. First contact

Before running a single transformation, work out what you are holding.

A real object arrives with history. Some of what is in `obs` is measurement,
some is a previous analyst's conclusion, and telling those apart is the first
skill this exercise tests.  *(Python track)*


In [ ]:
# ---------------------------------------------------------------------------
# The dataset downloads itself. Nothing to configure.
#
# If the download fails (institutional network, proxy, host down), fetch the
# file in a browser and upload it via the left sidebar: Files > Upload. The cell
# then picks it up automatically on rerun.
# ---------------------------------------------------------------------------
DATA_URL = "https://web-genobioinfo.toulouse.inrae.fr/~rekren/case_data.h5ad"
PATH     = "case_data.h5ad"

import os
if not os.path.exists(PATH):
    !wget -q --show-progress --tries=3 --timeout=60 -O {PATH} "{DATA_URL}"

if not os.path.exists(PATH) or os.path.getsize(PATH) < 1_000_000:
    if os.path.exists(PATH):
        os.remove(PATH)
    raise SystemExit(
        "Could not download the data. Fetch it in a browser from
  "
        + DATA_URL
        + "
then upload it via the left sidebar (Files > Upload) and rerun this cell."
    )
print(f"data file: {os.path.getsize(PATH) / 1e6:.1f} MB")

adata = sc.read_h5ad(PATH)

# Rename to a stable vocabulary, and mark the columns that are somebody else's
# conclusions rather than measurements.
ren = {"label": "condition", "stim": "condition",
       "replicate": "donor", "ind": "donor",
       "cell_type": "provided_cell_type", "cell": "provided_cell_type",
       "seurat_clusters": "provided_cluster"}
adata.obs = adata.obs.rename(columns={k: v for k, v in ren.items()
                                      if k in adata.obs.columns})
adata.obsm = {("provided_" + k.replace("X_", "")): v for k, v in adata.obsm.items()}

adata


In [ ]:
# What is in here? Extend these checks as much as you like.
print("cells x genes:", adata.shape, "\n")

print("--- metadata columns ---")
for c in adata.obs.columns:
    v = adata.obs[c]
    if v.dtype.kind in "fi" and v.nunique() > 20:
        print(f"  {c:24s} numeric   range {v.min():.1f} to {v.max():.1f}")
    else:
        print(f"  {c:24s} {v.nunique():2d} levels: "
              f"{', '.join(map(str, v.unique()[:8]))}")

print("\n--- are the values raw counts? ---")
X = adata.X
sub = X[:200]
d = sub.data if sp.issparse(sub) else np.asarray(sub).ravel()
print("integer-valued:", np.allclose(d, np.round(d)))
print("range of non-zero values:", d.min(), "to", d.max())
print("fraction of zeros:", round(1 - d.size / (sub.shape[0] * sub.shape[1]), 4))

print("\n--- the design ---")
print(adata.obs["condition"].value_counts())
print(pd.crosstab(adata.obs["donor"], adata.obs["condition"]))

print("\n--- embeddings already present ---")
print(list(adata.obsm.keys()))


### Q1
Describe what you have been given. Cover the dimensions, what a row and a column
represent, what the values in the matrix are, and how you *verified* they are
raw counts rather than something already transformed.

> **Your answer:**
>
> _(replace this line)_


### Q2
Go through the metadata columns one by one and sort them into three groups:

- **measured**: a property of the experiment or the sequencing
- **derived**: something computed from the counts, which you could recompute
- **concluded**: somebody's interpretation, which you could disagree with

Say which group each column falls into and how you decided. Then: which of these
columns would you be comfortable using in your own analysis, and which would you
set aside until you have checked them yourself?

> **Your answer:**
>
> _(replace this line)_


### Q3
The matrix is about 96 % zeros.

**(a)** Give two distinct reasons for that, one technical and one biological.

**(b)** **Translate from your own modality.** If your background is bulk
RNA-seq: a bulk count matrix has almost no zeros, and this one is almost all
zeros, yet both are counting the same molecules. Explain what changed. If your
background is proteomics: you already deal with missing values constantly.
Compare a zero here with a missing value in a proteomics run. Are they the same
kind of nothing? What follows for imputation?

> **Your answer:**
>
> _(replace this line)_


---

## Part 2. Quality control

Droplet data contains barcodes that are not single healthy cells: near-empty
droplets, dying cells, debris, and droplets that captured more than one cell.
This part is about deciding which barcodes to keep.

One warning before you start: at least one thing in this part will not match
what the tutorials tell you to do. Finding out what, and reasoning about it, is
the question.

*(Python track)*


In [ ]:
# Gene classes that carry diagnostic information about cell quality.
vn = adata.var_names.astype(str)

adata.var["mt"]   = vn.str.startswith("MT-")        # the standard first filter
adata.var["ribo"] = vn.str.startswith(("RPS", "RPL"))
adata.var["hb"]   = vn.str.match(r"^HB[ABDGQ]\d?$")

n_mt = int(adata.var["mt"].sum())
print("mitochondrial genes found:", n_mt)
print("ribosomal genes found:    ", int(adata.var["ribo"].sum()))
print("haemoglobin genes found:  ", int(adata.var["hb"].sum()), "\n")

if n_mt == 0:
    print(">>> Read that first number again, then answer Q5.")
    print("    Do not skip past this. It changes what you can do next.\n")

qc_vars = [v for v in ["mt", "ribo", "hb"] if int(adata.var[v].sum()) > 0]
sc.pp.calculate_qc_metrics(adata, qc_vars=qc_vars, percent_top=[20],
                           log1p=True, inplace=True)

cols = ["total_counts", "n_genes_by_counts"] + [f"pct_counts_{v}" for v in qc_vars]
adata.obs[cols].describe()


### Q4
For each metric below, say in one sentence what a cell with an extreme value of
it most likely is, and be specific about the direction.

| metric | what an extreme value most likely means |
|---|---|
| total counts per cell | *your answer* |
| genes detected per cell | *your answer* |
| ribosomal read fraction | *your answer* |
| haemoglobin read fraction | *your answer* |

Then: total counts and genes detected are strongly correlated. Why look at both?

> **Your answer:**
>
> _(replace this line)_


### Q5
Every standard quality-control workflow for this data type leans on one
particular metric, and you have just tried to compute it.

**(a)** Which metric, what does a high value of it indicate, and why is it the
usual first filter?

**(b)** What happened when you tried to compute it here? Report what you found,
not what you expected.

**(c)** Speculate about why. Give at least two possible explanations and say
which you find more plausible and why. You will not be able to confirm it, and
you are not expected to.

**(d)** Given that it is unavailable, what do you use instead? Name the
substitute metrics you will rely on and say what each one does and does not
capture that the missing one would have.

> **Your answer:**
>
> _(replace this line)_


In [ ]:
sc.pl.violin(adata, cols, groupby="condition", jitter=0.4,
             multi_panel=True, rotation=45)

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].scatter(adata.obs["total_counts"], adata.obs["n_genes_by_counts"],
              s=2, alpha=0.3)
ax[0].set_xlabel("total_counts"); ax[0].set_ylabel("n_genes_by_counts")
ax[1].hist(np.log10(adata.obs["total_counts"] + 1), bins=100)
ax[1].set_xlabel("log10(total_counts + 1)")
ax[2].hist(adata.obs["pct_counts_ribo"], bins=100)
ax[2].set_xlabel("pct_counts_ribo")
plt.tight_layout(); plt.show()


In [ ]:
# TODO: choose and justify your thresholds (Q6).
def mad_outlier(x, n_mads=5, log=False):
    x = np.log1p(np.asarray(x, float)) if log else np.asarray(x, float)
    m = np.median(x); d = np.median(np.abs(x - m))
    return (x < m - n_mads * d) | (x > m + n_mads * d)

MIN_GENES = None   # e.g. 200
MAX_GENES = None   # e.g. 2500
MAX_RIBO  = None   # e.g. 50

# keep = ...
# print(f"keeping {keep.sum()} of {adata.n_obs} cells")
# adata_raw = adata.copy()
# adata = adata[keep].copy()
# sc.pp.filter_genes(adata, min_cells=3)
# adata


### Q6
State every filtering threshold you applied and the evidence in the plots above
that led to each number. "The tutorial used it" is not evidence.

Then argue against yourself:

**(a)** How many cells did you remove, in absolute numbers and as a percentage?
Is that plausible?

**(b)** Which real immune cell type is most likely to be discarded by your
threshold? Think about cells with unusually little RNA.

**(c)** You probably also filtered genes. That reduces noise, but it is not
free. What do you lose?

> **Your answer:**
>
> _(replace this line)_


---

## Part 3. Normalisation and feature selection

Two cells can produce different total counts purely because of capture
efficiency rather than biology.

*(Python track)*


In [ ]:
# TODO: normalise and select variable genes (Q7, Q8).
adata.layers["counts"] = adata.X.copy()
N_HVG = None   # e.g. 2000

# sc.pp.normalize_total(adata, target_sum=1e4)
# sc.pp.log1p(adata)
# adata.raw = adata
# sc.pp.highly_variable_genes(adata, n_top_genes=N_HVG, flavor="seurat_v3",
#                             layer="counts", batch_key="condition")
# sc.pl.highly_variable_genes(adata)
# print(adata.var.query("highly_variable").sort_values("highly_variable_rank").head(30).index.tolist())


### Q7
**(a)** Explain what the normalisation you applied actually does, step by step.
If it has two steps, say what problem *each* one solves; they are different
problems.

**(b)** **Translate from your own modality.** From bulk RNA-seq: compare this
with CPM, TMM, or DESeq2 size factors. Name one assumption that carries over and
one that does not. From proteomics: compare it with how you normalise intensity
data across runs, and say which of the two situations you consider better posed
and why.

**(c)** A per-cell scaling factor assumes something about the cells being
compared. What does it assume, and can you think of a situation in immunology
where that assumption is clearly wrong?

> **Your answer:**
>
> _(replace this line)_


### Q8
**(a)** Why select a subset of genes at all? What specifically goes wrong if you
keep all of them?

**(b)** How many did you keep, and how did you land on that number? What would
change with 500 instead, or 5000?

**(c)** Look at the top-ranked variable genes. Do any of them look like they
track a technical or unwanted process rather than cell identity? Name them and
say what you suspect.

> **Your answer:**
>
> _(replace this line)_


---

## Part 4. Dimensionality reduction and clustering

You will produce your own embedding and your own clusters. The object already
contains an embedding and a clustering from the earlier analysis. Do not look at
them until you have made your own; then compare.

*(Python track)*


In [ ]:
# TODO: scale and run PCA (Q9).
# sc.pp.scale(adata, max_value=10)
# sc.tl.pca(adata, n_comps=50, svd_solver="arpack", random_state=SEED)
# sc.pl.pca_variance_ratio(adata, n_pcs=50, log=True)
# for pc in range(3):
#     load = pd.Series(adata.varm["PCs"][:, pc], index=adata.var_names)
#     print(f"PC{pc+1} +:", load.nlargest(10).index.tolist())
#     print(f"PC{pc+1} -:", load.nsmallest(10).index.tolist(), "\n")


### Q9
**(a)** Why run PCA before clustering rather than clustering the genes directly?
Give two distinct reasons.

**(b)** How many components did you carry forward, and on what basis? What is
the practical consequence of taking too few? Of taking too many?

**(c)** Look at the top gene loadings on the first two or three components. For
each, say what biological or technical process you think it is capturing, naming
the genes you are relying on.

> **Your answer:**
>
> _(replace this line)_


In [ ]:
# TODO: neighbours, UMAP, clustering at several resolutions (Q10).
N_PCS = None   # from Q9

# sc.pp.neighbors(adata, n_neighbors=15, n_pcs=N_PCS, random_state=SEED)
# sc.tl.umap(adata, random_state=SEED)
# for r in (0.2, 0.5, 1.0):
#     sc.tl.leiden(adata, resolution=r, key_added=f"leiden_{r}",
#                  random_state=SEED, flavor="igraph", n_iterations=2)
#     print(f"resolution {r}: {adata.obs[f'leiden_{r}'].nunique()} clusters")
# sc.pl.umap(adata, color=["leiden_0.2", "leiden_0.5", "leiden_1.0", "condition"],
#            ncols=2, size=6)


### Q10
**(a)** A UMAP is an embedding and Leiden is a clustering algorithm, and they run
on the same graph but are not the same thing. Explain the relationship. In
particular: if two clusters sit next to each other on the UMAP, does that mean
they are biologically similar?

**(b)** Which resolution did you settle on, and what evidence supports it?
Raising the resolution always yields more clusters, so what stops this being
arbitrary? Describe one concrete check you ran, or would run, to decide whether
a split is real.

> **Your answer:**
>
> _(replace this line)_


In [ ]:
# Compare your clustering with the one already in the object (Q11).
MY_CLUSTERS = None   # e.g. "leiden_0.5"

# print("--- your clusters vs condition ---")
# print(pd.crosstab(adata.obs[MY_CLUSTERS], adata.obs["condition"], normalize="index").round(2))
#
# print("\n--- provided clusters vs condition ---")
# print(pd.crosstab(adata.obs["provided_cluster"], adata.obs["condition"], normalize="index").round(2))
#
# print("\n--- your clusters vs provided clusters ---")
# print(pd.crosstab(adata.obs[MY_CLUSTERS], adata.obs["provided_cluster"]))
#
# from sklearn.metrics import adjusted_rand_score
# print("\nadjusted Rand index:",
#       round(adjusted_rand_score(adata.obs[MY_CLUSTERS], adata.obs["provided_cluster"]), 3))
#
# sc.pl.embedding(adata, basis="provided_umap", color=["condition", "provided_cell_type"], ncols=2)


### Q11
Now compare your clustering with the one already in the object.

**(a)** Do they agree? Quantify it, do not just eyeball the plots.

**(b)** Cross-tabulate your clusters against `condition`, and then the provided
clusters against `condition`. The two tables look very different. Describe the
difference precisely.

**(c)** **Speculate.** What must the earlier analyst have done that you did not?
You have enough information to work this out from the tables alone. State your
reasoning.

**(d)** Which of the two clusterings is *correct*? Argue it. This is a real
question, not a rhetorical one, and there is a defensible case on each side.

> **Your answer:**
>
> _(replace this line)_


---

## Part 5. The perturbation

This is the core of the exercise. Take your time here, at the cost of Part 4 or
Part 9 if necessary.

The dataset has two conditions and eight donors. Every donor contributes cells
to **both** conditions.

*(Python track)*


### Q12
Before running anything else in this part, write down the question you think
this experiment was designed to answer, based only on what you have seen so far.
One or two sentences. Do not edit it afterwards.

> **Your answer:**
>
> _(replace this line)_


In [ ]:
# Tools for Part 5. Use, modify or ignore.

# 1. Which genes separate the conditions within a single cell population?
# sub = adata[adata.obs["provided_cell_type"] == "<pick one>"].copy()
# sc.tl.rank_genes_groups(sub, groupby="condition", method="wilcoxon", use_raw=True)
# de = sc.get.rank_genes_groups_df(sub, group="stim")
# print(de.query("pvals_adj < 0.05").sort_values("logfoldchanges", ascending=False).head(25))

# 2. Score every cell for the genes you found, then look at where the score sits.
# my_genes = ["<gene>", "<gene>"]
# sc.tl.score_genes(adata, my_genes, score_name="programme")
# sc.pl.violin(adata, "programme", groupby="provided_cell_type",
#              rotation=90, stripplot=False)

# 3. The paired design: does the effect hold within every donor?
# (adata.obs.groupby(["donor", "condition"], observed=True)["programme"]
#           .mean().unstack())


### Q13
You now have to decide whether the separation between `ctrl` and `stim` is a
**technical batch effect that should be removed** or a **biological difference
that must be preserved**. Removing it in the first case is correct; removing it
in the second destroys the finding.

**(a)** Design the test *before* you run it. What would you measure, and what
result would point which way? Write this down first.

**(b)** Run it. Report what you found.

**(c)** The eight donors are the strongest tool you have here, because the
design is paired. Explain why, and use them.

**(d)** Commit to an answer and defend it with specific genes. Then say what
observation would have changed your mind.

> **Your answer:**
>
> _(replace this line)_


### Q14
**(a)** Name the genes that dominate the difference between conditions. Do they
form a coherent programme? If you recognise it, name it. If you do not, describe
what they have in common as far as you can tell from the data, and say how you
would identify it.

**(b)** Compute the effect separately within each cell type. Is the response the
same size everywhere? Report the ranking.

**(c)** **This is the most interesting result in the dataset.** One population
responds far less than the others. Identify it, and give your best explanation.
Speculate freely here, and say how confident you are.

> **Your answer:**
>
> _(replace this line)_


---

## Part 6. Cell identity

The object contains a `provided_cell_type` column. **Do your own annotation
first**, from marker genes, without looking at it. Then compare.

We will be able to tell from the notebook's execution order whether you did this
in the stated order, and answering honestly in Q17 matters more to us than
matching the provided labels.

*(Python track)*


### Q15
Describe your annotation procedure as you would in a methods section: how did you
get from a ranked gene list to a label?

Then fill in the table, one row per cluster, at your chosen resolution.

| cluster | n cells | your label | supporting genes (3 to 5) | confidence | what would raise it |
|---|---|---|---|---|---|
| | | | | | |

*(add rows as needed)*

> **Your answer:**
>
> _(replace this line)_


### Q16
**(a)** Which cluster are you least sure about, and what are the two competing
interpretations? Do not resolve the ambiguity artificially; describe it.

**(b)** Is there a cluster you suspect is not a real cell type at all? What
positive evidence would confirm that, and what would refute it?

**(c)** **Translate from your own modality.** From proteomics, and particularly
spatial proteomics: you are used to identifying populations from a small panel
of markers chosen in advance. Here you have every gene but a noisy, sparse
measurement of each. Compare the two situations. Which errors does each protect
you from, and which does each expose you to? From bulk RNA-seq: you are used to
deconvolution or signature scoring against a reference. How does assigning
identity cell by cell change what you can claim?

> **Your answer:**
>
> _(replace this line)_


### Q17
Now compare with `provided_cell_type`.

**(a)** Where do you agree and where do you differ? Quantify it.

**(b)** For each disagreement, say who you think is right and why.

**(c)** Did looking at the provided labels change your mind about any cluster?
Say so plainly if it did. Changing your mind on evidence is a good sign;
quietly rewriting an earlier answer is not.

> **Your answer:**
>
> _(replace this line)_


---

## Part 7. Differential expression done properly

Suppose you now want to report which genes respond to the perturbation in a
given cell type, in a form a colleague could act on.

*(Python track)*


In [ ]:
# Part 7. Cell-level test first, then aggregate.

# --- cell level ---
# sub = adata[adata.obs["provided_cell_type"] == "<pick one>"].copy()
# sc.tl.rank_genes_groups(sub, groupby="condition", method="wilcoxon", use_raw=True)
# de = sc.get.rank_genes_groups_df(sub, group="stim")
# print("genes with padj < 0.05:", int((de.pvals_adj < 0.05).sum()))
# print("number of donors:", adata.obs["donor"].nunique())

# --- aggregated to one profile per donor per condition ---
# import itertools
# rows, index = [], []
# X = adata.layers["counts"]
# for ct, dn, cond in itertools.product(sub.obs["provided_cell_type"].unique(),
#                                       adata.obs["donor"].unique(),
#                                       adata.obs["condition"].unique()):
#     m = ((adata.obs["provided_cell_type"] == ct) & (adata.obs["donor"] == dn)
#          & (adata.obs["condition"] == cond)).values
#     if m.sum() < 10: continue
#     rows.append(np.asarray(X[m].sum(axis=0)).ravel()); index.append((ct, dn, cond))
# pb = pd.DataFrame(rows, columns=adata.var_names,
#                   index=pd.MultiIndex.from_tuples(index, names=["cell_type","donor","condition"]))
# pb.shape


### Q18
The obvious approach is to take all cells of one type, split them by condition,
and run a test across individual cells.

**(a)** Do that, and report the number of genes passing a conventional
significance threshold.

**(b)** Now criticise the result you just produced. What is wrong with treating
each cell as an independent observation here? Name the specific statistical
problem.

**(c)** How does the number in (a) compare with the number of *donors*? What
does that comparison tell you?

> **Your answer:**
>
> _(replace this line)_


### Q19
**(a)** Aggregate the counts to one profile per donor per condition per cell
type, and analyse that instead. Report what changes.

**(b)** **Translate from your own modality.** If your background is bulk
RNA-seq, you have just rebuilt something very familiar. Say what it is, and why
the single-cell field spent years rediscovering it. If your background is
proteomics, describe the equivalent aggregation you would perform, and what your
replicate unit is.

**(c)** What did you lose by aggregating? Name something the cell-level analysis
could have told you that the aggregated one cannot.

> **Your answer:**
>
> _(replace this line)_


---

## Part 8. Synthesis

*(Python track)*


### Q20
In at most one page, state what is in this dataset: the populations, the
structure, and what you believe the two conditions represent. Write it for a
biologist who will act on it, not for a bioinformatician.

> **Your answer:**
>
> _(replace this line)_


### Q21
**(a)** Which single decision in this notebook had the largest influence on your
conclusions? How would they change if you had chosen differently?

**(b)** Which parts of your answer would you defend in a lab meeting, and which
are provisional?

**(c)** With a month instead of an afternoon, what would you do first, second and
third, and what question would each answer?

**(d)** List every assumption you made that you could not verify.

> **Your answer:**
>
> _(replace this line)_


### Q22
**(a)** Which external resources did you use, including documentation, forums and
language models, and for what specifically? Was there any output you accepted
without being able to verify it?

**(b)** Which cell did you run without fully understanding? There is almost
always one, and naming it scores better than pretending otherwise.

**(c)** Did you recognise the dataset? If so, at what point and from what? This
carries no penalty.

**(d)** What surprised you most, coming from your own area of omics? What was
harder than you expected, and what turned out to be easier?

> **Your answer:**
>
> _(replace this line)_


---

## Part 9. Presentation

Prepare **five to eight slides** for a 15 minute presentation to the group,
followed by discussion. Assume an audience of immunologists and one
bioinformatician.

Suggested shape, which you may ignore:

1. What the data was and how you approached it
2. Quality control, and the metric you could not compute
3. The populations you found
4. The perturbation: what it does, and to which cells most
5. One thing you are unsure about, and how you would resolve it

We would rather see one honest slide about uncertainty than five confident ones.

*(Python track)*


### Q23 (deliverable, not a written answer)
Attach the slides with your notebook. No template, any format.

> **Anything you want us to know about the slides:**
>
> _(replace this line)_


In [ ]:
# Environment record. Run this last and leave the output in place.
import session_info
try:
    session_info.show()
except Exception:
    from importlib.metadata import version
    for m in ["scanpy", "anndata", "numpy", "pandas", "scipy", "scikit-learn"]:
        try: print(m, version(m))
        except Exception: pass


---

## Before you send it back

- [ ] You worked through **one** section only, and every question in it has an answer.
- [ ] The notebook is saved **with outputs visible**, not cleared.
- [ ] The annotation table in Q15 is filled in.
- [ ] The slides from Part 9 are attached.

Thank you for the time you put into this.
